# Notebook 01: Raw Data Understanding & Exploratory Data Analysis (EDA)
## Nexora Climate Intelligence | CodeFest Datathon Finals 2026

### Purpose & Objectives
Before writing any data cleaning or modeling code, rigorous data science practices demand a complete understanding of the raw datasets:
1. **Schema & Dimensionality:** Column data types, shapes, and primary key candidates across all 5 raw datasets.
2. **Missing Values & Domain Rationale:** Diagnosing null patterns (e.g. atmospheric CO2 tracking at Mauna Loa vs regional sensors).
3. **Physical & Mathematical Bounds:** Verifying fuel mix 100% percentage closure and checking for negative carbon prices.
4. **Time Series Coverage:** Temporal spans and market liquidity across all carbon markets.
5. **Cross-Entity Consistency:** Auditing country overlap between emissions and energy generation profiles.


In [ ]:
# 1. Setup & Environment
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = BASE_DIR / 'raw'
print('Project root:', BASE_DIR.resolve())
print('Raw data dir:', RAW_DIR.resolve())


---
## 1. Raw Data Inventory & Dimension Overview
Let us inspect all five raw files provided by the Datathon organizers.


In [ ]:
raw_files = [
    'carbon_prices_daily.csv',
    'climate_events.csv',
    'co2_emissions_yearly.csv',
    'energy_mix_yearly.csv',
    'temperature_anomaly_monthly.csv'
]

inventory = []
dfs = {}
for fname in raw_files:
    fpath = RAW_DIR / fname
    df = pd.read_csv(fpath)
    dfs[fname] = df
    inventory.append({
        'Dataset': fname,
        'Rows': f'{len(df):,}',
        'Columns': df.shape[1],
        'Null Cells': int(df.isnull().sum().sum()),
        'Memory (MB)': round(df.memory_usage(deep=True).sum() / (1024 * 1024), 2)
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)


---
## 2. Dataset 1: Daily Carbon Prices (`carbon_prices_daily.csv`)
Analyzing the 5 major emissions trading systems: EU_ETS, RGGI, California, UK_ETS, and China_ETS.


In [ ]:
cp = dfs['carbon_prices_daily.csv'].copy()
cp['date'] = pd.to_datetime(cp['date'])

print('=== Carbon Prices Market Breakdown ===')
market_stats = cp.groupby('market').agg(
    start_date=('date', 'min'),
    end_date=('date', 'max'),
    trading_days=('price', 'count'),
    min_price=('price', 'min'),
    median_price=('price', 'median'),
    max_price=('price', 'max'),
    std_price=('price', 'std')
).reset_index()
display(market_stats)

print('Negative price count:', (cp['price'] < 0).sum())
print('Duplicate (market, date) pairs:', cp.duplicated(subset=['market', 'date']).sum())


In [ ]:
# Visualizing Daily Carbon Prices Across Markets
fig, ax = plt.subplots(figsize=(14, 6))
for m, g in cp.groupby('market'):
    ax.plot(g['date'], g['price'], label=m, alpha=0.85, linewidth=1.5)

ax.set_title('Daily Carbon Price Trends Across 5 ETS Markets (2005 - 2026)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Price (Local Currency / EUR)', fontsize=11)
ax.legend(title='ETS Market')
plt.tight_layout()
plt.show()


---
## 3. Datasets 2 & 3: Annual CO2 Emissions & Energy Mix
Checking country coverage, primary keys, and fuel percentage closure.


In [ ]:
co2 = dfs['co2_emissions_yearly.csv'].copy()
energy = dfs['energy_mix_yearly.csv'].copy()

print(f"CO2 Emissions: {co2['country'].nunique()} countries, Years {co2['year'].min()} to {co2['year'].max()}")
print(f"Energy Mix: {energy['country'].nunique()} countries, Years {energy['year'].min()} to {energy['year'].max()}")

co2_keys = set(zip(co2['iso3'], co2['year']))
energy_keys = set(zip(energy['iso3'], energy['year']))
print('CO2 Keys:', len(co2_keys), '| Energy Keys:', len(energy_keys))
print('Shared Matching Keys:', len(co2_keys.intersection(energy_keys)), '(100% Exact 1-to-1 Match!)')


In [ ]:
# Fuel Mix Percentage Closure Verification
fuel_cols = ['coal_pct', 'oil_pct', 'gas_pct', 'nuclear_pct', 'hydro_pct', 'solar_pct', 'wind_pct', 'other_renewables_pct']
fuel_sums = energy[fuel_cols].sum(axis=1)

print('=== Fuel Share Sum Verification ===')
print(f'Minimum row fuel sum: {fuel_sums.min():.4f}%')
print(f'Maximum row fuel sum: {fuel_sums.max():.4f}%')
print(f'Rows within [99.98%, 100.02%]: {((fuel_sums >= 99.98) & (fuel_sums <= 100.02)).sum()} / {len(energy)}')


---
## 4. Dataset 4: Major Climate & Policy Events (`climate_events.csv`)
Auditing 50 high-impact events across regions, severity ratings, and categories.


In [ ]:
events = dfs['climate_events.csv'].copy()
events['date'] = pd.to_datetime(events['date'])

print('=== Climate Events Breakdown ===')
display(events['event_type'].value_counts().to_frame('Event Count'))

print('\nSeverity Score Distribution (1-10 Scale):')
display(events['severity_score'].describe().to_frame('Severity Stats'))


---
## 5. Dataset 5: Monthly Temperature Anomaly & CO2 PPM (`temperature_anomaly_monthly.csv`)
Auditing regional temperature anomalies and explaining the 2,212 missing values in atmospheric CO2 concentration.


In [ ]:
temp = dfs['temperature_anomaly_monthly.csv'].copy()

print('=== Temperature Anomaly Regional Breakdown ===')
temp_summary = temp.groupby('region').agg(
    months=('year_month', 'count'),
    min_anomaly=('temp_anomaly_deg_c', 'min'),
    mean_anomaly=('temp_anomaly_deg_c', 'mean'),
    max_anomaly=('temp_anomaly_deg_c', 'max'),
    valid_co2_ppm=('co2_ppm', 'count')
).reset_index()
display(temp_summary)

print('\nDOMAIN JUSTIFICATION ON MISSING co2_ppm:')
print('co2_ppm is non-null ONLY for region == "Global" (316 monthly observations).')
print('Scientific Rationale: Atmospheric CO2 is tracked globally at Mauna Loa Observatory, not regionally.')


---
## 6. Synthesis & Cleaning Blueprint for Notebook 02
Based on this raw data inspection, here is the exact protocol to execute in **Notebook 02**:
1. **Merge CO2 + Energy Mix:** Perform inner join on `(iso3, year)` to produce `country_clean.csv` (1,350 rows, 0 nulls). Engineer composite transition features (`clean_baseload_pct`, `fossil_ratio`).
2. **Zero-Leakage Price Features:** Sort `carbon_prices_daily.csv` chronologically per market. Add cyclical calendar variables and strictly shifted autoregressive lags (1 to 30 days) and rolling windows (7d, 30d).
3. **Standardize Events:** Parse dates and validate binary flags in `events_clean.csv`.
4. **Preserve Temperature Scope:** Retain regional anomalies in `temp_clean.csv` and document global scope of `co2_ppm`.
